# L05 · Agregar con groupby

| | |
|---|---|
| **Qué vas a aprender** | split-apply-combine sin perder el norte |
| **Cuánto tiempo** | 60 min |
| **Datasets** | `ventas_2024_este.csv`, `ventas_2024_oeste.tsv` |

> **Al terminar vas a poder** responder "cuánto vendimos por región este año" y saber
> si el número que sale es real o un artefacto de tu `groupby`.

In [ ]:
import sys
from pathlib import Path

# JupyterLab arrancado en la raíz del proyecto deja el cwd ahí.
# Si abres el notebook con el cwd en notebooks/, sube un nivel.
ROOT = Path.cwd()
if not (ROOT / "datos").exists() and (ROOT.parent / "datos").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks"))
DATOS = ROOT / "datos"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from herramientas import (cargar, mostrar, ficha, guardar_csv, guardar_excel,
                           guardar_figura, parsear_importes, parsear_fecha,
                           clave_canonica, detectar_alfabeto, auditar_fusiones,
                           resumen_entrega, perfil_memoria, contar_falsos_nulos)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
plt.style.use("seaborn-v0_8-whitegrid")
pd.__version__

In [ ]:
df = pd.read_csv(
    DATOS / "ventas_2024_este.csv", sep=";", decimal=",", encoding="latin-1"
)
df["Fecha Pedido"] = pd.to_datetime(df["Fecha Pedido"], format="%d/%m/%Y")
df["Importe"] = parsear_importes(df["Importe"])
df["Descuento %"] = pd.to_numeric(df["Descuento %"], errors="coerce")

print(df.shape)
mostrar(df, 5, "con los tipos ya convertidos")

## El concepto

`groupby` son tres pasos: **split** (trocear), **apply** (aplicar una función a cada
trozo) y **combine** (pegar los resultados).

Lo que la gente se salta es el paso 3. `groupby` **no** muta el DataFrame: devuelve un
objeto `DataFrameGroupBy` que espera a que le pidas el resultado. Por eso
`df.groupby("Region")` no imprime nada, y por eso `df.groupby("Region").mean()` te
promete el promedio de **todas** las columnas numéricas, incluidas las que no te
interesan.

## Manos a la obra

In [ ]:
# agregacion simple
df.groupby("Region")["Importe"].sum().sort_values(ascending=False)

In [ ]:
# varios indicadores de golpe, con agg y un diccionario
cierre = df.groupby("Region").agg(
    ventas=("Importe", "sum"),
    pedidos=("ID Pedido", "nunique"),
    ticket_medio=("Importe", "mean"),
    descuento_medio=("Descuento %", "mean"),
)

mostrar(cierre.sort_values("ventas", ascending=False), titulo="CIERRE POR REGIÓN")

In [ ]:
# dos niveles: mes x region, con unstack para dejarlo en ancho
df["mes"] = df["Fecha Pedido"].dt.to_period("M").astype(str)
mensual = df.groupby(["mes", "Region"])["Importe"].sum().unstack(fill_value=0)
mostrar(mensual.head(6), titulo="MES x REGIÓN")

## ⚠️ La trampa

### `size` vs `count`: el bug que sobrevive a todos los code reviews

`size` cuenta **filas**. `count` cuenta **valores no nulos**. Si tu columna tiene nulos,
los dos dan números distintos, y el que uses cambia el número de tu informe.

In [ ]:
d = pd.DataFrame({
    "grupo": ["A", "A", "A", "B", "B"],
    "valor": [1.0, np.nan, 3.0, 4.0, 5.0],
})

print("size  :", d.groupby("grupo").size().to_dict())
print("count :", d.groupby("grupo")["valor"].count().to_dict())
print("nuniq :", d.groupby("grupo")["valor"].nunique().to_dict())

In [ ]:
# En ESTE extracto cada fila es un pedido: size y nunique coinciden.
# El caso en que no coinciden es una tabla de líneas.
print("filas por region  :", df.groupby("Region").size().sum())
print("pedidos distintos :", df["ID Pedido"].nunique())

lineas = pd.DataFrame({
    "pedido": ["P1", "P1", "P1", "P2"],
    "region": ["Norte", "Norte", "Norte", "Sur"],
})
print("lineas (size)     :", lineas.groupby("region").size().to_dict())
print("pedidos (nunique) :", lineas.groupby("region")["pedido"].nunique().to_dict())

En `ventas_2024_este.csv` hay una fila por pedido, así que `size` y
`nunique` dan el mismo número. Si el ERP emitiera una fila por línea y un pedido
tuviera 3 líneas, `size` contaría líneas y `nunique` contaría pedidos. El ejemplo
de cuatro filas de arriba es ese caso: Norte tiene 3 líneas y 1 pedido.

### La trampa 2: `apply` devolviendo un DataFrame

Cuando la lambda de `apply` devuelve un DataFrame en vez de un escalar, pandas
reconstruye la forma y las filas se multiplican. Si quieres las primeras filas de cada
grupo, filtra **antes** de agrupar.

In [ ]:
# MAL: apply devolviendo un df reconstruye la forma
malo = df.groupby("Region").apply(
    lambda g: g[["Importe"]].head(2), include_groups=False
)
print("filas con apply-head:", len(malo), "(hay", df["Region"].nunique(), "regiones)")

# BIEN: filtra antes de agrupar
umbral = df["Importe"].quantile(0.95)
bueno = df[df["Importe"] > umbral]
print("filas filtrando antes:", len(bueno))

## 🧪 Tu turno

- **Ejercicio 1** — Top 5 categorías por facturación, con su nº de pedidos y su ticket
  medio. Pista: usa `.nlargest(5)` sobre una agregación, no sobre el DataFrame original.
- **Ejercicio 2** — Qué categoría tiene mayor dispersión de importes (coeficiente de
  variación = desviación / media).
- **Ejercicio 3** — Calcula el % que cada región representa sobre el total de su mes, y
  comprueba que la suma por mes da 100% (redondeo incluido).

## 📦 Workbook integrador: cierre mensual

**Contexto.** El director comercial pide, cada mes, un Excel con ventas, nº de pedidos y
ticket medio por región y categoría. El problema real no es el `groupby`: es que **los
dos ficheros del ERP no comparten esquema** y hay que unirlos primero.

**Tu trabajo.**

1. Cargar los dos ficheros (`.csv` y `.tsv`) y renombrar sus columnas a un esquema
   común.
2. Unirlos en un único DataFrame de ventas de todo el año.
3. Producir el cierre mensual × región × categoría.
4. Exportarlo a `salida/L05_cierre_mensual.xlsx` **con formato**: cabecera con fondo,
   anchos ajustados y primera fila congelada.

In [ ]:
# paso 1: el TSV tiene otro esquema. lo mapeamos al mismo.
oeste = pd.read_csv(DATOS / "ventas_2024_oeste.tsv", sep="\t", encoding="utf-8")
oeste = oeste.rename(columns={
    "order_id": "ID Pedido",
    "order_date": "Fecha Pedido",
    "customer_id": "ID Cliente",
    "region": "Region",
    "category": "Categoria",
    "qty": "Unidades",
    "unit_price": "Precio Unitario",
    "total": "Importe",
})
oeste["Fecha Pedido"] = pd.to_datetime(oeste["Fecha Pedido"], format="%Y-%m-%d")
oeste["Descuento %"] = 0.0   # el ERP oeste no lo manda: queda documentado

comunes = [c for c in df.columns if c in oeste.columns]
ventas = pd.concat([df[comunes], oeste[comunes]], ignore_index=True)

print("este:", df.shape, "| oeste:", oeste.shape, "-> unidas:", ventas.shape)
print("columnas comunes:", comunes)

In [ ]:
# paso 2: el cierre
ventas["mes"] = ventas["Fecha Pedido"].dt.to_period("M").astype(str)

cierre = (
    ventas.groupby(["mes", "Region", "Categoria"])
           .agg(ventas=("Importe", "sum"),
                pedidos=("ID Pedido", "nunique"),
                unidades=("Unidades", "sum"))
           .reset_index()
)
cierre["ticket_medio"] = (cierre["ventas"] / cierre["pedidos"]).round(2)
cierre["%_del_mes"] = (
    cierre["ventas"] / cierre.groupby("mes")["ventas"].transform("sum") * 100
).round(2)

mostrar(cierre, 8, "CIERRE MENSUAL")

In [ ]:
# la comprobación que de verdad importa: los porcentajes tienen que sumar 100
suma_pct = cierre.groupby("mes")["%_del_mes"].sum().round(1).unique()
print("suma de %_del_mes por mes:", suma_pct)
assert all(abs(x - 100) < 0.5 for x in suma_pct), "los porcentajes no cuadran"

In [ ]:
# paso 3: el Excel con formato
ruta_xlsx = guardar_excel(cierre, "L05_cierre_mensual",
                          hoja="Cierre", formatear_importe="#,##0.00 €")
ruta_csv = guardar_csv(cierre, "L05_cierre_mensual")
resumen_entrega(ruta_xlsx, ruta_csv)

### Criterios de aceptación

1. `ventas` tiene las filas de los **dos** ficheros sumadas (12.000 + 9.000 = 21.000).
2. `cierre` tiene columnas `mes`, `Region`, `Categoria`, `ventas`, `pedidos`,
   `unidades`, `ticket_medio`, `%_del_mes`.
3. La aserción del `assert` pasa: los porcentajes suman 100 ±0,5 en todos los meses.
4. Se generan `salida/L05_cierre_mensual.xlsx` y `.csv`, y el `.xlsx` abre con la
   cabecera formateada y la primera fila congelada.